In [2]:
# 1. Import the libraries
import pandas as pd
from datasets import load_dataset

# 2. Download the IMDB Movie Reviews dataset using the full namespace
print("Downloading the IMDB dataset...")
dataset = load_dataset("stanfordnlp/imdb")

# 3. Convert the training data into a Pandas DataFrame so we can read it easily
df_train = pd.DataFrame(dataset['train'])

# 4. Look at the first 5 rows
df_train.head()

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


In [3]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# 1. Download the required NLTK dictionaries
nltk.download('stopwords')
nltk.download('wordnet')

# 2. Initialize our NLP tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# 3. The Master Cleaning Function
def clean_text(text):
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove everything that isn't a letter and make lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text).lower()

    # Split into individual words
    words = text.split()

    # Keep the word if it's not a stop word, and lemmatize it
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    # Stitch the words back together into a single string
    return ' '.join(cleaned_words)

# 4. Test it on the first 5 rows
print("Testing the cleaning algorithm...\n")
df_train_test = df_train.head().copy()
df_train_test['cleaned_text'] = df_train_test['text'].apply(clean_text)

# 5. Show the Before and After
for i in range(2):
    print(f"--- REVIEW {i+1} ---")
    print(f"ORIGINAL: {df_train_test['text'].iloc[i][:150]}...")
    print(f"CLEANED:  {df_train_test['cleaned_text'].iloc[i][:150]}...\n")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


Testing the cleaning algorithm...

--- REVIEW 1 ---
ORIGINAL: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard th...
CLEANED:  rented curiousyellow video store controversy surrounded first released also heard first seized u custom ever tried enter country therefore fan film co...

--- REVIEW 2 ---
ORIGINAL: "I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be ta...
CLEANED:  curious yellow risible pretentious steaming pile doesnt matter one political view film hardly taken seriously level claim frontal male nudity automati...



In [4]:
from collections import Counter
import numpy as np

# 1. Clean the entire Training set
print("Cleaning all 25,000 training reviews... (this will take a minute)")
df_train['cleaned_text'] = df_train['text'].apply(clean_text)

# 2. Grab and clean the Test set (we need this later to see if our model actually works)
print("Cleaning all 25,000 testing reviews...")
df_test = pd.DataFrame(dataset['test'])
df_test['cleaned_text'] = df_test['text'].apply(clean_text)

# 3. Build the Master Vocabulary
print("Building the vocabulary dictionary...")
# Combine all words from the training set into one massive list
all_words = ' '.join(df_train['cleaned_text']).split()
word_counts = Counter(all_words)

# Keep only the top 10,000 most common words to keep the model fast and ignore typos
VOCAB_SIZE = 10000
common_words = [word for word, count in word_counts.most_common(VOCAB_SIZE)]

# Create the mapping dictionary (Starting at 1, because 0 is reserved for padding)
word_to_int = {word: i+1 for i, word in enumerate(common_words)}

print("\n--- VOCABULARY STATS ---")
print(f"Total unique words found: {len(word_counts)}")
print(f"Vocabulary restricted to top {VOCAB_SIZE} words.")
print(f"Integer ID for 'movie': {word_to_int.get('movie')}")
print(f"Integer ID for 'terrible': {word_to_int.get('terrible')}")
print(f"Integer ID for 'brilliant': {word_to_int.get('brilliant')}")

Cleaning all 25,000 training reviews... (this will take a minute)
Cleaning all 25,000 testing reviews...
Building the vocabulary dictionary...

--- VOCABULARY STATS ---
Total unique words found: 99737
Vocabulary restricted to top 10000 words.
Integer ID for 'movie': 1
Integer ID for 'terrible': 288
Integer ID for 'brilliant': 413


In [5]:
# 1. The Encoding Function
def text_to_ints(text, word_to_int):
    # Convert words to IDs. If a word isn't in our top 10,000, we just skip it.
    return [word_to_int[word] for word in text.split() if word in word_to_int]

print("Translating English to Integers...")
train_encoded = [text_to_ints(text, word_to_int) for text in df_train['cleaned_text']]
test_encoded = [text_to_ints(text, word_to_int) for text in df_test['cleaned_text']]

# 2. The Padding Function
SEQ_LENGTH = 200 # Standardizing all reviews to 200 words

def pad_features(reviews_ints, seq_length):
    # Create a giant matrix of zeros: (Number of reviews x Sequence Length)
    features = np.zeros((len(reviews_ints), seq_length), dtype=int)

    for i, row in enumerate(reviews_ints):
        if len(row) > 0:
            # Drop the integers into the end of the zero matrix (pre-padding)
            features[i, -len(row):] = np.array(row)[:seq_length]

    return features

print(f"Padding all sequences to {SEQ_LENGTH} tokens...")
train_padded = pad_features(train_encoded, SEQ_LENGTH)
test_padded = pad_features(test_encoded, SEQ_LENGTH)

# 3. Extract the Labels (0 for Negative, 1 for Positive)
train_labels = np.array(df_train['label'])
test_labels = np.array(df_test['label'])

# 4. Verify the Architecture
print("\n--- MATRIX STATS ---")
print(f"Training Features Matrix: {train_padded.shape}")
print(f"Training Labels Matrix: {train_labels.shape}")
print("\nLook at Review #1 (Notice the zeros padding the front, followed by the word IDs):")
print(train_padded[0])

Translating English to Integers...
Padding all sequences to 200 tokens...

--- MATRIX STATS ---
Training Features Matrix: (25000, 200)
Training Labels Matrix: (25000,)

Look at Review #1 (Notice the zeros padding the front, followed by the word IDs):
[   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0 1414  266  891 5833 3038   22
  518   19  441   22   79 6088   48  655 2268  470 1421  114    2 1017
 2769   14   13   39 5348   98   95 3493  336  693  640 3890   57  703
  189   30  686   57  746  551  161  281  479  731   99  657  864  732
 2417  201 1207  732 2142  607 1986 3418 1735  480 2193  294  336 1272
 6011  873  245  259   42  484 1017 7341   14  294  869   15  143    9
  137    4 5916   27 3983  222   56 142

In [6]:
import torch
import torch.nn as nn

class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, drop_prob=0.5):
        super().__init__()
        self.output_dim = output_dim
        self.n_layers = n_layers
        self.hidden_dim = hidden_dim

        # 1. Embedding Layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # 2. LSTM Layer (batch_first=True means our matrix has batches as the first dimension)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, n_layers,
                            dropout=drop_prob, batch_first=True)

        # 3. Dropout Layer (prevents the model from memorizing the training data)
        self.dropout = nn.Dropout(drop_prob)

        # 4. Fully Connected & Sigmoid Layer
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, hidden):
        batch_size = x.size(0)

        # Convert integers to embeddings
        embeds = self.embedding(x)

        # Pass through LSTM
        lstm_out, hidden = self.lstm(embeds, hidden)

        # We only care about the LSTM's output from the VERY LAST word in the sequence
        # because that's when it has read the entire review.
        lstm_out = lstm_out[:, -1, :]

        # Pass through dropout and fully connected layer
        out = self.dropout(lstm_out)
        out = self.fc(out)

        # Squash to a probability between 0 and 1
        sig_out = self.sigmoid(out)

        return sig_out, hidden

    def init_hidden(self, batch_size, device):
        # Initializes the LSTM's memory to zero at the start of a new batch
        weight = next(self.parameters()).data
        hidden = (weight.new(self.n_layers, batch_size, self.hidden_dim).zero_().to(device),
                  weight.new(self.n_layers, batch_size, self.hidden_dim).zero_().to(device))
        return hidden

# Instantiate the model architecture
VOCAB_SIZE_PLUS_PAD = VOCAB_SIZE + 1 # +1 for the 0 padding
EMBEDDING_DIM = 400
HIDDEN_DIM = 256
OUTPUT_DIM = 1
N_LAYERS = 2

# Build the network
model = SentimentLSTM(VOCAB_SIZE_PLUS_PAD, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS)
print(model)

SentimentLSTM(
  (embedding): Embedding(10001, 400)
  (lstm): LSTM(400, 256, num_layers=2, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [8]:
from torch.utils.data import TensorDataset, DataLoader

# 1. Setup the Hardware
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# Move the model we built earlier to the hardware
model = model.to(device)

# 2. Create the DataLoaders
train_data = TensorDataset(torch.from_numpy(train_padded), torch.from_numpy(train_labels).float())
test_data = TensorDataset(torch.from_numpy(test_padded), torch.from_numpy(test_labels).float())

# Feed the data in batches of 50
BATCH_SIZE = 50
train_loader = DataLoader(train_data, shuffle=True, batch_size=BATCH_SIZE, drop_last=True)
test_loader = DataLoader(test_data, shuffle=False, batch_size=BATCH_SIZE, drop_last=True)

# 3. Define the Loss and Optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 4. The Training Loop
EPOCHS = 3

print("\nStarting Training...")
for epoch in range(EPOCHS):
    model.train() # Put model in training mode
    h = model.init_hidden(BATCH_SIZE, device) # Reset memory

    total_loss = 0
    correct_predictions = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Detach hidden states so we don't backpropagate through the entire history
        h = tuple([each.data for each in h])

        # Reset the gradients
        model.zero_grad()

        # 1. Forward Pass (Make a guess)
        output, h = model(inputs, h)

        # ---> THE FIX: Squeeze the [50, 1] output to a flat [50] array <---
        output = output.squeeze()

        # 2. Calculate the Error
        loss = criterion(output, labels)
        total_loss += loss.item()

        # Track accuracy (round 0.8 to 1.0, and 0.2 to 0.0)
        predicted = torch.round(output)
        correct_predictions += (predicted == labels).sum().item()

        # 3. Backward Pass (Learn from the mistake)
        loss.backward()

        # 4. Update the weights
        optimizer.step()

    # Print stats at the end of each Epoch
    avg_loss = total_loss / len(train_loader)
    accuracy = (correct_predictions / (len(train_loader) * BATCH_SIZE)) * 100

    print(f"Epoch: {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Training Accuracy: {accuracy:.2f}%")

print("\nTraining Complete! Brain is fully baked.")

Training on: cpu

Starting Training...
Epoch: 1/3 | Loss: 0.6105 | Training Accuracy: 65.83%
Epoch: 2/3 | Loss: 0.4249 | Training Accuracy: 80.86%
Epoch: 3/3 | Loss: 0.3042 | Training Accuracy: 87.77%

Training Complete! Brain is fully baked.


In [9]:
# 1. Put the model in evaluation mode (turns off dropout)
model.eval()

test_losses = []
num_correct = 0

print("Testing the model on 25,000 blind reviews...")

# 2. Turn off gradients to save memory and speed up testing
with torch.no_grad():
    h = model.init_hidden(BATCH_SIZE, device)

    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        h = tuple([each.data for each in h])

        # Make a prediction
        output, h = model(inputs, h)

        # Apply the exact same squeeze fix here!
        output = output.squeeze()

        # Calculate loss
        test_loss = criterion(output, labels)
        test_losses.append(test_loss.item())

        # Calculate accuracy
        pred = torch.round(output)
        num_correct += (pred == labels).sum().item()

# 3. Print Final Stats
print(f"Test Loss: {np.mean(test_losses):.4f}")
test_acc = (num_correct / (len(test_loader) * BATCH_SIZE)) * 100
print(f"Final Test Accuracy: {test_acc:.2f}%")

# 4. EXPORT THE BRAIN
torch.save(model.state_dict(), 'sentiment_lstm.pth')
print("\nSuccess! Model saved as 'sentiment_lstm.pth'")

Testing the model on 25,000 blind reviews...
Test Loss: 0.3254
Final Test Accuracy: 86.61%

Success! Model saved as 'sentiment_lstm.pth'


In [10]:
import json

# Export the vocabulary mapping
with open('vocab.json', 'w') as f:
    json.dump(word_to_int, f)

print("Success! Vocabulary saved as 'vocab.json'")

Success! Vocabulary saved as 'vocab.json'
